# 05 - Learned Embeddings (Part B)

**Author:** Alisha | **Course:** Introduction to NLP, Assignment 2 (Project Phase 2)

This notebook trains three learned word-embedding models on the cleaned IMDB reviews from Phase 1 and compares them:

- **B1:** Word2Vec CBOW
- **B2:** Word2Vec Skip-gram
- **B3:** how the two training objectives differ
- **B4:** fastText and an out-of-vocabulary (OOV) demonstration

All models use identical hyperparameters so that only the training scheme differs.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable when the notebook runs from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.embeddings import (
    PARAMS, OOV_WORDS, load_sentences, train_cbow, train_skipgram,
    train_fasttext, oov_demo, save_models,
)

## Data and hyperparameters

Sentences are the cleaned review texts from `data/processed/imdb_cleaned.csv` (column `text_clean`), split into tokens.

In [2]:
sentences = load_sentences()
n_docs = len(sentences)
n_tokens = sum(len(s) for s in sentences)
corpus_vocab = len({w for s in sentences for w in s})
print(f"Documents          : {n_docs}")
print(f"Total tokens       : {n_tokens}")
print(f"Unique tokens      : {corpus_vocab}")
print(f"Average doc length : {n_tokens / n_docs:.1f} tokens")
print("Shared hyperparameters:", PARAMS)

Documents          : 49582
Total tokens       : 6219655
Unique tokens      : 98695
Average doc length : 125.4 tokens
Shared hyperparameters: {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5, 'seed': 42, 'workers': 4}


## B1. Word2Vec CBOW (`sg=0`)

CBOW predicts the **centre** word from the average of its surrounding **context** words.

In [3]:
cbow, t_cbow = train_cbow(sentences)
print("Model              : Word2Vec CBOW (sg=0)")
print("Hyperparameters    :", PARAMS)
print(f"Training runtime   : {t_cbow:.2f} s")
print(f"Vocabulary size    : {len(cbow.wv)}")

Model              : Word2Vec CBOW (sg=0)
Hyperparameters    : {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5, 'seed': 42, 'workers': 4}
Training runtime   : 28.20 s
Vocabulary size    : 38984


## B2. Word2Vec Skip-gram (`sg=1`)

Skip-gram uses exactly the same hyperparameters; only the training objective changes.

In [4]:
skipgram, t_sg = train_skipgram(sentences)
print("Model              : Word2Vec Skip-gram (sg=1)")
print("Hyperparameters    :", PARAMS)
print(f"Training runtime   : {t_sg:.2f} s")
print(f"Vocabulary size    : {len(skipgram.wv)}")

Model              : Word2Vec Skip-gram (sg=1)
Hyperparameters    : {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5, 'seed': 42, 'workers': 4}
Training runtime   : 90.39 s
Vocabulary size    : 38984


In [5]:
# Summary of B1 and B2 (Screenshot 5)
import pandas as pd
summary = pd.DataFrame({
    "Model": ["Word2Vec CBOW (sg=0)", "Word2Vec Skip-gram (sg=1)"],
    "vector_size": [PARAMS["vector_size"]] * 2,
    "window": [PARAMS["window"]] * 2,
    "min_count": [PARAMS["min_count"]] * 2,
    "epochs": [PARAMS["epochs"]] * 2,
    "seed": [PARAMS["seed"]] * 2,
    "Training time (s)": [round(t_cbow, 2), round(t_sg, 2)],
    "Vocabulary size": [len(cbow.wv), len(skipgram.wv)],
})
summary

,Model,vector_size,window,min_count,epochs,seed,Training time (s),Vocabulary size
0,Word2Vec CBOW (sg=0),100,5,5,5,42,28.20,38984
1,Word2Vec Skip-gram (sg=1),100,5,5,5,42,90.39,38984


## B3. CBOW vs Skip-gram

| | CBOW | Skip-gram |
|---|---|---|
| Training objective | context words -> centre word | centre word -> context words |
| Predictions per window | one (context is averaged) | one per context word |
| Speed | faster | slower |
| Strength | frequent words | rare words |

**CBOW (context -> centre).** For every position the surrounding words are averaged into one vector, which is used to predict the centre word. Because there is only one prediction per window, training is fast, and the averaging smooths the signal, which suits **frequent** words that appear in many contexts.

**Skip-gram (centre -> context).** The centre word is used to predict each surrounding word separately, so every (centre, context) pair becomes its own training example. A rare word therefore receives many more updates than it would under CBOW, where it is averaged away inside a context. This is why Skip-gram usually represents **rare movie terms** (for example specific director, genre or technical words) better, at the cost of slower training.

*Check against our results:* the runtimes above should show CBOW training faster than Skip-gram. Whether Skip-gram also gave better neighbours for rare words is checked in the next cell and in Part C.

## B4. fastText and OOV handling

Word2Vec treats each word as an indivisible unit, so a word missing from the vocabulary has no vector. **fastText** represents a word as the sum of its **character n-grams** (for example `cinematography` includes pieces such as `<ci`, `cin`, `ine`, `nem`, ...), so it can build a vector for an unseen or misspelled word from the pieces it shares with known words.

In [6]:
fasttext, t_ft = train_fasttext(sentences)
print("Model              : fastText (character n-grams)")
print("Hyperparameters    :", PARAMS)
print(f"Training runtime   : {t_ft:.2f} s")
print(f"Vocabulary size    : {len(fasttext.wv)}")

Model              : fastText (character n-grams)
Hyperparameters    : {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5, 'seed': 42, 'workers': 4}
Training runtime   : 117.72 s
Vocabulary size    : 38984


In [7]:
# OOV / misspelled word demonstration (Screenshot 6)
oov_demo(cbow, skipgram, fasttext, OOV_WORDS)


--- OOV / misspelled word: 'cinematographyyy' ---
Word2Vec CBOW: KeyError (word not in vocabulary)
Word2Vec Skip-gram: KeyError (word not in vocabulary)
fastText in vocabulary: False
fastText top-5 similar words (via character n-grams):
   cinematography       0.9942
   cinematograph        0.9930
   cinematographic      0.9431
   cinematographer      0.9295
   cinematographers     0.9180

--- OOV / misspelled word: 'horrrorrr' ---
Word2Vec CBOW: KeyError (word not in vocabulary)
Word2Vec Skip-gram: KeyError (word not in vocabulary)
fastText in vocabulary: False
fastText top-5 similar words (via character n-grams):
   horror               0.8951
   horrors              0.8709
   horrorfest           0.8119
   horrid               0.7461
   terror               0.7215


In [8]:
# Extra evidence for B3: neighbours of in-vocabulary words in all three models
probe_words = ["movie", "acting", "horror", "director"]
for w in probe_words:
    print(f"\n=== {w} ===")
    for name, m in (("CBOW", cbow), ("Skip-gram", skipgram), ("fastText", fasttext)):
        try:
            neighbours = [n for n, _ in m.wv.most_similar(w, topn=5)]
            print(f"{name:<10}: {', '.join(neighbours)}")
        except KeyError:
            print(f"{name:<10}: '{w}' not in vocabulary (min_count={PARAMS['min_count']})")


=== movie ===
CBOW      : film, movies, flick, really, films
Skip-gram : film, atleast, sequal, suprise, hellborn
fastText  : imovie, moviegoer, telemovie, film, moovie

=== acting ===
CBOW      : performances, casting, directing, direction, writing
Skip-gram : directing, sequencing, varies, staggeringly, complimented
fastText  : enacting, overacting, reenacting, exacting, detracting

=== horror ===
CBOW      : slasher, exploitation, genre, zombie, splatter
Skip-gram : slasher, slashers, omnibus, amicus, goriest
fastText  : horrorfest, horrors, slasher, error, slashers

=== director ===
CBOW      : filmmaker, directors, filmmakers, screenwriter, altman
Skip-gram : writer, screenwriter, cinematographer, directors, filmmaker
fastText  : directory, directorial, directv, direct, directors


In [9]:
# Save models locally for Parts C and D (models/ is NOT committed to git)
save_models(cbow, skipgram, fasttext)


Models saved in /home/lisha/NLP-Assignment-01-pipeline/models (not committed to git).
